# Phase 4 – 1D CNN Multi-Class Classification (NSL-KDD)

## Objective
Implement a 1D Convolutional Neural Network for 4-class intrusion detection
and compare performance with classical ML baselines.

## Dataset
NSL-KDD (preprocessed)
Input shape: (N, 122)
Reshaped for CNN: (N, 1, 122)

## Evaluation Metrics
- Accuracy
- Macro F1-score
- Classification Report


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report

base_path = "/content/drive/MyDrive/nsl_kdd/"

X_train = np.load(base_path + "X_train.npy")
X_test  = np.load(base_path + "X_test.npy")
y_train = np.load(base_path + "y_train.npy")
y_test  = np.load(base_path + "y_test.npy")

# Compute class weights for imbalance handling
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

X_train_tensor = X_train_tensor.unsqueeze(1)
X_test_tensor  = X_test_tensor.unsqueeze(1)

print("Reshaped:", X_train_tensor.shape)

class NSLKDDDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 256

train_dataset = NSLKDDDataset(X_train_tensor, y_train_tensor)
test_dataset  = NSLKDDDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class CNN1D(nn.Module):
    def __init__(self, num_classes=4):
        super(CNN1D, self).__init__()

        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)

        self.pool = nn.MaxPool1d(2)
        self.dropout = nn.Dropout(0.5)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x)
        x = x.view(x.size(0), -1)

        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)

        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Convert to torch tensor and move to device
# class_weights = torch.tensor(class_weights, dtype=torch.float32)

unique, counts = np.unique(y_train, return_counts=True)
print("Class distribution:", dict(zip(unique, counts)))

from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print("Computed class weights:", class_weights)

model = CNN1D(num_classes=4).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Using device:", device)

num_epochs = 12

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {running_loss/len(train_loader):.4f} "
          f"Train Accuracy: {train_acc:.2f}%")
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print("CNN Results:")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Macro F1:", f1_score(all_labels, all_preds, average='macro'))
print(classification_report(all_labels, all_preds))

cnn_acc = accuracy_score(all_labels, all_preds)
cnn_f1  = f1_score(all_labels, all_preds, average='macro')

results = pd.DataFrame({
    "Model": ["1D CNN (Smoothed Weights)"],
    "Accuracy": [cnn_acc],
    "Macro_F1": [cnn_f1],
    "Dropout": [0.5],
    "Epochs": [12],
    "Batch_Size": [256],
    "Loss_Function": ["CrossEntropy (Smoothed Weights)"],
    "Device": [str(device)]
})

results.to_csv("/content/drive/MyDrive/nsl_kdd/cnn_final_results.csv", index=False)

print("Results saved successfully.")

report = classification_report(all_labels, all_preds, output_dict=True)
report_df = pd.DataFrame(report).transpose()

report_df.to_csv("/content/drive/MyDrive/nsl_kdd/cnn_classification_report.csv")

print("Classification report saved.")

torch.save(model.state_dict(), "/content/drive/MyDrive/nsl_kdd/cnn_model_final.pth")

print("Model saved successfully.")




Train shape: (125973, 122)
Test shape: (22544, 122)
Reshaped: torch.Size([125973, 1, 122])
Using device: cuda
Class distribution: {np.int64(0): np.int64(45927), np.int64(1): np.int64(67343), np.int64(2): np.int64(11656), np.int64(3): np.int64(1047)}
Computed class weights: tensor([ 0.6857,  0.4677,  2.7019, 30.0795], device='cuda:0')
Using device: cuda
Epoch [1/12] Loss: 0.3165 Train Accuracy: 90.91%
Epoch [2/12] Loss: 0.1202 Train Accuracy: 95.87%
Epoch [3/12] Loss: 0.0988 Train Accuracy: 96.28%
Epoch [4/12] Loss: 0.0918 Train Accuracy: 96.55%
Epoch [5/12] Loss: 0.0729 Train Accuracy: 97.10%
Epoch [6/12] Loss: 0.0796 Train Accuracy: 96.96%
Epoch [7/12] Loss: 0.0686 Train Accuracy: 97.31%
Epoch [8/12] Loss: 0.0652 Train Accuracy: 97.35%
Epoch [9/12] Loss: 0.0587 Train Accuracy: 97.57%
Epoch [10/12] Loss: 0.0572 Train Accuracy: 97.62%
Epoch [11/12] Loss: 0.0598 Train Accuracy: 97.47%
Epoch [12/12] Loss: 0.0607 Train Accuracy: 97.39%
CNN Results:
Accuracy: 0.76166607523066
Macro F1: 0.71

## Final CNN Configuration

- Dropout: 0.5
- Epochs: 12
- Loss Function: CrossEntropyLoss with smoothed class weights [1,1,3,10]
- Device: GPU (CUDA)

### Results:
- Accuracy: 0.7376
- Macro F1-score: 0.6635
- Improved minority detection compared to baseline
- Reduced overfitting

Conclusion:
Smoothed class-weighting provided the best balance between overall accuracy and minority attack detection.


## Inference Latency Measurement (GPU)


In [ ]:
import time

model.eval()

# Warmup
with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        _ = model(inputs)
        break

torch.cuda.synchronize()
start_time = time.time()

with torch.no_grad():
    for inputs, _ in test_loader:
        inputs = inputs.to(device)
        _ = model(inputs)

torch.cuda.synchronize()
end_time = time.time()

total_time = end_time - start_time
num_samples = len(test_dataset)

time_per_sample = total_time / num_samples
throughput = num_samples / total_time

print(f"Total inference time: {total_time:.4f} seconds")
print(f"Average time per sample: {time_per_sample*1000:.6f} ms")
print(f"Throughput: {throughput:.2f} samples/sec")

latency_results = pd.DataFrame({
    "Mode": ["Batch_256"],
    "Total_Inference_Time_sec": [total_time],
    "Time_per_Sample_ms": [time_per_sample * 1000],
    "Throughput_samples_per_sec": [throughput],
    "Batch_Size": [batch_size],
    "Device": [str(device)]
})

latency_results.to_csv(
    "/content/drive/MyDrive/nsl_kdd/cnn_latency_batch256_phase4.csv",
    index=False
)

print("Latency results saved.")

Total inference time: 0.1793 seconds
Average time per sample: 0.007952 ms
Throughput: 125760.62 samples/sec
Latency results saved.
